# Lecture 20: Six joints

**The challenge.** Lecture 19 solved a two-link arm exactly. Real arms have six
joints. The state goes from 6 numbers to 18, the limits from 4 to 12, and the number
of ways to ride them at once from 8 to 728. None of the theory changes, so the
question is purely practical: what breaks first when the same code meets a bigger
problem?

The way to answer that is to measure, so this lecture starts with the measurements.
The fixes they led to are two ideas: keep a robot's dynamics as a **program** rather
than a formula, and solve for the riding controls **at run time** rather than in closed
form. It uses the gradients, Jacobians and reverse-mode derivatives of the
{doc}`prerequisite lecture <p1_jacobians_and_hessians>`.

## 1. What broke

Each row was the next obstacle once the one above was out of the way. The times are
from one 16-core machine, and are a guide rather than a benchmark.

| What was measured | Before | After | What it took |
|---|---|---|---|
| deriving one arm's conditions | 400 s at 3 joints, never finished at 6 | a few seconds at 6 joints | a tape (Section 3) |
| kernels per set of limits ridden | one each, for 728 sets | one for all of them | solving at run time (Section 4) |
| compiling a long Jacobian | 159 s for one function | about 20 s | cutting it into pieces |
| the search for the structure | stalled on overlapping arcs | 91% of the motions surveyed (Section 6) | three fixes in the search, and correct second derivatives |

One thing that did *not* break is worth saying. With every limit active, one call to
the compiled dynamics takes about 4 microseconds and one Jacobian about 25: fast
enough that the arithmetic was never the problem.

## 2. A robot in a loop

In 1980 J. Y. S. Luh, M. W. Walker and R. P. C. Paul published an algorithm for a
robot's torques that was fast enough to run while the robot moved: the **recursive
Newton–Euler algorithm**. Velocities and accelerations are passed from the base
out to the hand, one link at a time, and then forces and moments are passed back from
the hand to the base. The work grows linearly with the number of joints, where the
Lagrangian formulations then in use grew far faster.

The package has it in `robot.joint_torques`, working on SymPy symbols. Each link is one
row of a table.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d

from auto_optimal_control import examples
from auto_optimal_control.build import build_kernels
from auto_optimal_control.homotopy import Homotopy
from auto_optimal_control.pontryagin import derive
from auto_optimal_control.robot import GRAVITY, SIX_JOINT_LINKS, joint_torques, link_frames
from auto_optimal_control.tape import Tape

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"

q, w, a = (sp.symbols(f"{s}1:7", real=True) for s in "qwa")
tau = joint_torques(SIX_JOINT_LINKS, q, w, a, GRAVITY)
print("one link:", SIX_JOINT_LINKS[1])
print("operations in each torque, written out:", [sp.count_ops(e) for e in tau])

The first torque has about a hundred thousand operations. But the algorithm did not
do a hundred thousand operations to compute it. Written out, every intermediate result
is copied into everywhere it is used; the recursion computes it once.

## 3. Keep the program: the tape

A `Tape` records each distinct subexpression once, as a straight-line program, and
it applies the rules of differentiation to the program. This is the automatic
differentiation of the prerequisite lecture.

In [ ]:
tape = Tape()
roots = [tape.from_sympy(e) for e in tau]
print("steps for all six torques:", len(tape.reachable(roots)))

names = [s.name for s in (*q, *w, *a)]
start = time.perf_counter()
gradients = [tape.gradient(r, names) for r in roots]
print(f"all 108 first derivatives: {len(tape.reachable(sum(gradients, [])))} steps, "
      f"in {time.perf_counter() - start:.3f} s")

# Check against a difference of the formula as SymPy wrote it.
f = sp.lambdify((q, w, a), tau, "numpy")
rng = np.random.default_rng(0)
Q, W, A = rng.uniform(-1, 1, (3, 6))
values = dict(zip(names, np.concatenate([Q, W, A])))
step = 1e-6
difference = (np.array(f(Q + step * np.eye(6)[2], W, A), dtype=float)
              - np.array(f(Q - step * np.eye(6)[2], W, A), dtype=float)) / (2 * step)
print("d tau / d q3, tape vs difference:", np.abs(
    np.array(tape.evaluate([g[2] for g in gradients], values)) - difference).max())

**Nothing changes for the user.** You still write the limit as a SymPy expression. When
`derive` meets a path constraint longer than 200 operations, it records it on a tape and
puts a small opaque function, `Outlined`, in its place. SymPy can differentiate that
function: it asks the tape, and gets either another opaque function or zero. So
the rest of the derivation still sees short expressions. The two limits on one joint
share one *root*, because they are $\pm\tau_i$ minus a constant.

In [ ]:
problem = examples.six_joint_arm()
start = time.perf_counter()
pmp = derive(problem)
print(f"derive: {time.perf_counter() - start:.1f} s")
print("which root each limit uses:", pmp.riding.root_of)
print("and with what sign        :", pmp.riding.sign_of)
print("the first limit is now    :", str(pmp.path_constraints[0].expr)[:40], "...")
print("ways to ride limits at once:", len(pmp.path_constraints) + len(pmp.boundary_sets),
      "= 3^6 - 1: each joint at its upper limit, its lower one, or neither")

## 4. Solve when needed: one kernel for every set

Lecture 14 found the control that rides a limit by solving the riding conditions
symbolically, once per set of limits. Six limits at once would mean inverting a matrix
of mass-matrix entries in closed form, 728 times over.

But those equations are linear in the unknowns. For limits with
$\dot g_i = A_i(x)\,u + b_i(x)$ and a Hamiltonian quadratic in $u$, the controls and
multipliers solve

$$
\begin{bmatrix} R & A^\top \\ A & 0 \end{bmatrix}
\begin{bmatrix} u \\ \mu \end{bmatrix}
=
\begin{bmatrix} -\,\partial H/\partial u\,|_{u=0} \\ -b \end{bmatrix},
\qquad R = H_{uu},
$$

and C can solve that where it is needed, for whichever limits are active. The arc's
Jacobian, which the state transition matrix needs, comes from differentiating the same
system. Writing $y = (u, \mu)$ and calling these equations $r(z, y) = 0$,

$$
\frac{\partial y}{\partial z} = -K^{-1} \frac{\partial r}{\partial z},
$$

where $K$ is the matrix above, already factored. The second derivatives of the limits
enter only through one number, $\sum_k \mu_k \dot h_k$ over the roots $h_k$, whatever the
number of active limits. A single reverse pass gives its gradient, which is why the
Jacobian takes so few steps.

So `riding.py` generates one set of functions, and the arc is chosen by an argument. A
useful check, independent of how it was derived: along an arc, the limits it rides
should stop changing.

In [ ]:
start = time.perf_counter()
k = build_kernels(pmp)
print(f"compiling, or loading from the cache: {time.perf_counter() - start:.0f} s")

params = np.zeros(len(pmp.params))
params[0], params[-6:] = 1.0, 50.0             # T, and the torque limits
z = np.random.default_rng(1).uniform(-1, 1, k.N_Z)
step = 1e-6
for arc in (0, k.boundary_set([0, 3]), k.boundary_set([0, 2, 4, 6, 8, 10])):
    dz = k.boundary_rhs(arc, 0.3, z, params)
    rate = (k.constraint(0.3, z + step * dz, params) - k.constraint(0.3, z - step * dz, params)) / (2 * step)
    ridden = list(k.arc_constraints(arc))
    print(f"riding {len(ridden)}: largest rate among those ridden {np.abs(rate[ridden]).max():.1e}; "
          f"among the others {np.abs(np.delete(rate, ridden)).max():.1e}")
print("(riding one side of all six joints fixes every torque, so the other sides stop too)")

## 5. The arm solved

The same `Homotopy` call as Lecture 15, with no problem-specific code. The motion goes
from $q = 0$ to a pose with all six joints moved, in one second. Gravity sets a floor:
it holds joint 2 at 54 N·m just standing at the start pose, so a limit below that makes
the problem infeasible from $t = 0$, and `solve` says so immediately. Each limit below
is therefore set part of the way from the larger end-point torque to the peak of the
unconstrained motion, here 90% of the way, on all six joints at once.

In [ ]:
h = Homotopy(problem)
T, qf = 1.0, np.array([1.0, -1.0, 0.8, 1.0, -0.8, 1.0])
s = np.linspace(0, 1, 801)[:, None]
torque_of = lambda Q, W, A: np.array([np.broadcast_to(np.asarray(v, dtype=float), Q.shape[:1])
                                      for v in f(Q.T, W.T, A.T)]).T
unconstrained = torque_of(qf * (10 * s ** 3 - 15 * s ** 4 + 6 * s ** 5),
                          qf * (30 * s ** 2 - 60 * s ** 3 + 30 * s ** 4) / T,
                          qf * (60 * s - 180 * s ** 2 + 120 * s ** 3) / T ** 2)
peak = np.abs(unconstrained).max(axis=0)
ends = np.maximum(np.abs(unconstrained[0]), np.abs(unconstrained[-1]))
box = ([-50] * 18, [50] * 18)

loose = h.solve([T, *np.zeros(6), *qf, *(1.5 * peak)], box)
print(f"limits loose: J = {loose.objective:.9f}; six quintics give {360 * qf @ qf / T ** 5:.9f}")

limits = ends + 0.9 * (peak - ends)
start = time.perf_counter()
r = h.solve([T, *np.zeros(6), *qf, *limits], box)
print(f"all six limited: exact = {r.exact}, J = {r.objective:.6f}, in {time.perf_counter() - start:.1f} s")
print("the search's events:", [f"{kind} {name}" for kind, name in r.arcs][:6], "...")

impossible = h.solve([T, *np.zeros(6), *qf, *(0.5 * limits)], box)
print("with half those limits:", impossible.message[:80])

**An independent check.** As in Lecture 19, the jerks are replayed through plain SciPy
integration, and the torques recomputed from the formula SymPy wrote, without the
kernels or the tape.

In [ ]:
tr = r.shooting.trajectory(r.solution.v, n_points=20001)
t, u = tr["t"], tr["u"]
keep = np.concatenate([[True], np.diff(t) > 0])
jerk = interp1d(t[keep], u[keep], axis=0)
sol = solve_ivp(lambda ti, x: np.concatenate([x[6:], jerk(min(ti, t[-1]))]), (0, t[-1]), np.zeros(18),
                rtol=1e-10, atol=1e-12, max_step=1e-3, t_eval=np.linspace(0, t[-1], 2001))
torque = torque_of(sol.y[:6].T, sol.y[6:12].T, sol.y[12:].T)
replayed = float(np.trapezoid(0.5 * (u[keep] ** 2).sum(axis=1), t[keep]))
print(f"cost: {r.objective:.9f} vs replay {replayed:.9f}")
print(f"final state error: {np.abs(sol.y[:, -1] - np.concatenate([qf, np.zeros(12)])).max():.1e}")
print(f"largest torque over its limit: {(np.abs(torque) - limits).max():.1e}")

fig, axes = plt.subplots(2, 3, figsize=(11.5, 5.2), sharex=True, constrained_layout=True)
for i, ax in enumerate(axes.flat):
    ax.plot(sol.t, torque[:, i], color=BLUE, lw=2)
    ax.plot(s[:, 0] * T, unconstrained[:, i], color=GRAY, lw=1, ls="--")
    for sign in (1, -1):
        ax.axhline(sign * limits[i], color=ORANGE, lw=1)
    ax.set_title(f"joint {i + 1}")
    ax.spines[["top", "right"]].set_visible(False)
axes[0, 0].plot([], [], color=BLUE, lw=2, label="the solution")
axes[0, 0].plot([], [], color=GRAY, lw=1, ls="--", label="without limits")
axes[0, 0].plot([], [], color=ORANGE, lw=1, label="the limit")
axes[0, 0].legend(frameon=False, fontsize=8)
for ax in axes[1]:
    ax.set_xlabel("t")
for ax in axes[:, 0]:
    ax.set_ylabel("torque")
plt.show()

*Each joint's torque. Without limits, each joint's largest torque would pass its limit; the solution rides the limits there, with arcs overlapping. Every joint's motion changes a little because of the others' limits, which couple through the mass matrix. This is a stress test: a real arm's last joint does not have a limit near 1 mN·m.*

In [ ]:
from IPython.display import HTML
from matplotlib import animation

plt.rcParams["animation.embed_limit"] = 30
n_frames = 60
idx = np.linspace(0, len(sol.t) - 1, n_frames).astype(int)
riding = np.abs(np.abs(torque) - limits) < 1e-4
points = []
for i in idx:
    frames = link_frames(SIX_JOINT_LINKS, sol.y[:6, i])
    points.append(np.vstack([np.zeros(3), [p for _, p in frames]]))
points = np.array(points)
fig = plt.figure(figsize=(5.2, 4.6), dpi=72)
ax = fig.add_subplot(projection="3d")
ax.plot(*points[:, -1].T, color="k", lw=0.7, ls=":")
line, = ax.plot([], [], [], "-o", color=BLUE, lw=4, ms=5)
hot, = ax.plot([], [], [], "o", color=ORANGE, ms=10)
label = ax.text2D(0.02, 0.95, "", transform=ax.transAxes, fontsize=9)
span = 0.85
ax.set(xlim=(-span, span), ylim=(-span, span), zlim=(-0.3, 1.0), xticks=[], yticks=[], zticks=[])


def draw(n):
    p = points[n]
    line.set_data(p[:, 0], p[:, 1])
    line.set_3d_properties(p[:, 2])
    on = riding[idx[n]]
    hot.set_data(p[:-1, 0][on], p[:-1, 1][on])
    hot.set_3d_properties(p[:-1, 2][on])
    label.set_text(f"t = {sol.t[idx[n]]:.2f}   joints at a limit: {list(np.flatnonzero(on) + 1)}")
    return line, hot, label


anim = animation.FuncAnimation(fig, draw, frames=n_frames, interval=90, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

*The arm, moving: press play. A joint turns orange while it rides a torque limit. The dotted line is the path of the hand.*

## 6. How often does it work?

One worked example says little about a search. So the solve was run on random motions,
each with a random goal and duration, $T \in [0.8, 1.5]$ and $q_f \in [-1.5, 1.5]^6$,
at four limit tightnesses, with a 60-second budget on one machine running eight
solves at once.

| Limits tightened | Motions solved exactly | Median time when solved |
|---|---|---|
| joints 1–3 (the arm), 95% to 70% | 87 of 96 (91%) | 3 s |
| all six (also the wrist, at 80%) | 49 of 93 (53%) | 32 s |

Most of the failures in the second row were out of time, or close to it. A retry of 15 of
them with a 300-second budget solved 7, the slowest in 160 seconds, so 300 seconds is
now the default budget when arcs are solved at run time. The other failures stall well
before the budget runs out.

The other half of a search's quality is what it does on failure. A failure here is
reported as `exact = False`, never as a wrong answer, because a result is only accepted
when it passes the checks of Lecture 15.

The search needed three repairs that had nothing to do with the size of the arithmetic.
All three only appear when many arcs overlap:

- a violation that spanned the junction of another arc was reported as two violations,
  so the search tried to enter one limit twice;
- a tiny violation at a tangential junction, at the level of the path tolerance, sent
  the search round in a circle, re-activating the arc it already had;
- a new arc born inside another must be guessed with its events in the right order, and
  the solve only converges from the right one. Nested is the commonest.

A fourth problem was a wrong derivative in the new code, of the kind the prerequisite
lecture warns about. It was found by comparing the Jacobian with finite differences.

## 7. What this does not do yet

- **Bounded jerks.** The run-time kernel solves an equality-constrained system, with no
  bounds on $u$. With bounds it becomes a small quadratic program whose active set changes
  along the arc. So minimum time on six joints, the problem of Lecture 19 at full scale,
  is reported as unsupported rather than attempted.
- **Limits of order 2 at this size,** such as a velocity limit on a joint of the six-joint
  arm. Small problems still get them from the symbolic route.
- **A better search for many short arcs.** The harder half of the survey is where the
  remaining failures are, and where the remaining time goes.
- **Global optimality.** As everywhere in this course, these are extremals: they satisfy
  the necessary conditions. A six-joint arm can have several.

Learned warm starts remain a possible last step. What this lecture shows is that they
were not needed to reach six joints, and that the costs they would target are now
the search's, not the arithmetic's.